In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
!pip install keras-core --upgrade
!pip install -q keras-nlp --upgrade

# This sample uses Keras Core, the multi-backend version of Keras.
# The selected backend is TensorFlow (other supported backends are 'jax' and 'torch')
import os
os.environ['KERAS_BACKEND'] = 'tensorflow'

In [ ]:
import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
import tensorflow as tf
import keras_core as keras
import keras_nlp
from sklearn.metrics import ConfusionMatrixDisplay, confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt

print("TensorFlow version:", tf.__version__)
print("KerasNLP version:", keras_nlp.__version__)

##　トレーニングデータの読み込み

In [ ]:
df_train = pd.read_csv("/kaggle/input/nlp-getting-started/train.csv")
df_test = pd.read_csv("/kaggle/input/nlp-getting-started/test.csv")

print('Training Set Shape = {}'.format(df_train.shape))
print('Training Set Memory Usage = {:.2f} MB'.format(df_train.memory_usage().sum() / 1024**2))
print('Test Set Shape = {}'.format(df_test.shape))
print('Test Set Memory Usage = {:.2f} MB'.format(df_test.memory_usage().sum() / 1024**2))

In [ ]:
df_train.head()

##　テストデータの読み込み

In [ ]:
df_test.head()

In [ ]:
df_train["length"] = df_train["text"].apply(lambda x : len(x))
df_test["length"] = df_test["text"].apply(lambda x : len(x))

print("Train Length Stat")
print(df_train["length"].describe())
print()

print("Test Length Stat")
print(df_test["length"].describe())

In [ ]:
BATCH_SIZE = 32
NUM_TRAINING_EXAMPLES = df_train.shape[0]
TRAIN_SPLIT = 0.8
VAL_SPLIT = 0.2
STEPS_PER_EPOCH = int(NUM_TRAINING_EXAMPLES)*TRAIN_SPLIT // BATCH_SIZE

EPOCHS = 2
AUTO = tf.data.experimental.AUTOTUNE

In [ ]:
from sklearn.model_selection import train_test_split

X = df_train["text"]
y = df_train["target"]

X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=VAL_SPLIT, random_state=42)

X_test = df_test["text"]

In [ ]:
# Load a DistilBERT model.
preset= "distil_bert_base_en_uncased"

# Use a shorter sequence length.
preprocessor = keras_nlp.models.DistilBertPreprocessor.from_preset(preset,
                                                                   sequence_length=160,
                                                                   name="preprocessor_4_tweets"
                                                                  )

# Pretrained classifier.
classifier = keras_nlp.models.DistilBertClassifier.from_preset(preset,
                                                               preprocessor = preprocessor, 
                                                               num_classes=2)

classifier.summary()

In [ ]:
from tensorflow.keras.optimizers import Adam

classifier.compile(
    loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    optimizer=Adam(1e-5),
    metrics=["accuracy"]
)

In [ ]:
def displayConfusionMatrix(y_true, y_pred, dataset):
    disp = ConfusionMatrixDisplay.from_predictions(
        y_true,
        np.argmax(y_pred, axis=1),
        display_labels=["Not Disaster","Disaster"],
        cmap=plt.cm.Blues
    )

    tn, fp, fn, tp = confusion_matrix(y_true, np.argmax(y_pred, axis=1)).ravel()
    f1_score = tp / (tp+((fn+fp)/2))

    disp.ax_.set_title("Confusion Matrix on " + dataset + " Dataset -- F1 Score: " + str(f1_score.round(2)))

In [ ]:
y_pred_train = classifier.predict(X_train)

displayConfusionMatrix(y_train, y_pred_train, "Training")

In [ ]:
y_pred_val = classifier.predict(X_val)

displayConfusionMatrix(y_val, y_pred_val, "Validation")

In [ ]:
#改善前

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix, classification_report, f1_score
from scipy.sparse import hstack


# 📥 データ読み込み
df = pd.read_csv('/kaggle/input/nlp-getting-started/train.csv')

# 🔧 テキストとラベル
X_text = df['text']
y = df['target']

# ✅ 文字数特徴量を作成
X_len = X_text.apply(len).values.reshape(-1, 1)

# 🔠 TF-IDFベクトル化
vectorizer = TfidfVectorizer(stop_words='english', max_features=10000)
X_tfidf = vectorizer.fit_transform(X_text)

# ✅ TF-IDFと文字数を結合
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
X_len_scaled = scaler.fit_transform(X_len)
X_combined = hstack([X_tfidf, X_len_scaled])

# ✂️ 学習用とテスト用に分割
X_train, X_test, y_train, y_test = train_test_split(X_combined, y, test_size=0.2, random_state=42)

# 🧠 モデル学習
model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)

# 📊 予測
y_pred = model.predict(X_test)

# ✅ 混同行列とF1スコア
cm = confusion_matrix(y_test, y_pred)
f1 = f1_score(y_test, y_pred, average='weighted')

# 📈 可視化
plt.figure(figsize=(7,6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Not Disaster', 'Disaster'],
            yticklabels=['Not Disaster', 'Disaster'])
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title(f'Confusion Matrix (with length feature)\nF1 Score (weighted): {f1:.3f}', fontsize=14)
plt.tight_layout()
plt.show()

# 🧾 詳細な評価
print(classification_report(y_test, y_pred, target_names=['Not Disaster', 'Disaster']))

In [ ]:
#改善後

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import re
import nltk
from nltk.stem import WordNetLemmatizer
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix, classification_report, f1_score
from scipy.sparse import hstack, csr_matrix

# --- 1. データ読み込みと前処理強化 ---

# データの読み込み
df = pd.read_csv('/kaggle/input/nlp-getting-started/train.csv')

# NLTKリソースの準備（初回実行時はダウンロードが必要です）
# nltk.download('punkt')
# nltk.download('wordnet')
lemmatizer = WordNetLemmatizer()

# 🧹 テキストをクリーニングする関数
def clean_text(text):
    # URLを除去
    text = re.sub(r'https?://\S+|www\.\S+', '', text)
    # HTMLタグを除去
    text = re.sub(r'<.*?>', '', text)
    # ユーザー名（@mention）を除去
    text = re.sub(r'@\w+', '', text)
    # ハッシュタグの記号 '#' のみ除去（単語は残す）
    text = re.sub(r'#', '', text)
    # 句読点や数字などの非アルファベット文字を除去
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    # 小文字に変換
    text = text.lower()
    # トークン化とレンマ化
    tokens = nltk.word_tokenize(text)
    lemmatized_tokens = [lemmatizer.lemmatize(word) for word in tokens]
   
    return " ".join(lemmatized_tokens)

# データフレームにクリーニング済みテキストを適用
df['cleaned_text'] = df['text'].apply(clean_text)


# --- 2. 特徴量エンジニアリングの高度化 ---

# 🔑 keywordカラムの欠損値を'nokeyword'で埋める
df['keyword'] = df['keyword'].fillna('nokeyword')

# keywordとクリーニング済みテキストを結合
df['combined_text'] = df['keyword'] + " " + df['cleaned_text']

# 文字数を特徴量として追加
df['text_length'] = df['combined_text'].apply(len)

# ✂️ テキスト、数値特徴量、ラベルを準備
X_text = df['combined_text']
X_length = df['text_length'].values.reshape(-1, 1)
y = df['target']

# 🔠 TF-IDFベクトル化 (N-gramを導入)
vectorizer = TfidfVectorizer(
    ngram_range=(1, 2),  # 1-gramと2-gramの両方を特徴量として使用
    stop_words='english',
    max_features=15000   # 特徴量を少し増やして多くの情報を捉える
)
X_vec = vectorizer.fit_transform(X_text)

# 文字数特徴量を疎行列化して結合
X_length_sparse = csr_matrix(X_length)
X_all = hstack([X_vec, X_length_sparse])


# --- 3. モデル学習と評価 ---

# 🧪 訓練データとテストデータに分割
X_train, X_test, y_train, y_test = train_test_split(X_all, y, test_size=0.2, random_state=42, stratify=y)

# 🧠 モデル学習 (元のロジスティック回帰を使用)
model = LogisticRegression(max_iter=1000, random_state=42)
model.fit(X_train, y_train)

# 📊 予測確率取得（災害クラスの確率）
proba = model.predict_proba(X_test)[:, 1]

# 通常の予測ラベルを取得
y_pred = model.predict(X_test)

# テストセットの文字数を取得（結合した特徴量の最後の列）
X_test_lengths = X_test[:, -1].toarray().reshape(-1)

# --- 後処理: 元のコードと同じロジックを適用 ---
threshold = 0.5
y_pred_adjusted = y_pred.copy()
for i, length in enumerate(X_test_lengths):
    # 文字数が少ないツイート（例: 80字以下）で、
    # かつ災害と予測される確率が低い場合、非災害に修正
    if length <= 80 and proba[i] < threshold:
        y_pred_adjusted[i] = 0  # 非災害(0)に修正

# ✅ 混同行列
cm = confusion_matrix(y_test, y_pred_adjusted)

# ✅ F1スコア（weighted平均）
f1 = f1_score(y_test, y_pred_adjusted, average='weighted')

# 📈 混同行列プロット
plt.figure(figsize=(7, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Not Disaster', 'Disaster'],
            yticklabels=['Not Disaster', 'Disaster'])
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title(f'Confusion Matrix (with Post-processing)\nF1 Score (weighted): {f1:.4f}', fontsize=14)
plt.tight_layout()
plt.show()

# 🧾 詳細な分類レポート
print("--- Classification Report (with Post-processing) ---")
print(classification_report(y_test, y_pred_adjusted, target_names=['Not Disaster', 'Disaster']))